<h1 align = "Center">
Lotus Group Customer Value & Profitability Analysis

<h2 align = "Center">
Feature Engineering
</h2>
</h1>

In this notebook we turn the cleaned sales data into the two tables our machine-learning components need:

| Table | One row per | Used for |
|---|---|---|
| `customer_features.csv` | Customer | **Customer value segmentation** (K-Means clustering) |
| `order_features.csv` | Sales order | **Loss-making order detection** (classification) |

The exploratory analysis shaped our choices:

- Revenue is very skewed and closely tied to Electronics share, so the customer table needs **margin and mix** measures, not just spend.
- Every loss-making order contained a deeply discounted Electronics item, so the order table needs **discount and basket-mix** measures.

Both tables are saved to a new `01_Data/features/` folder, so the modelling notebooks can start directly from them.

# Setup and Data Loading

In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../01_Data/processed")
features_path = Path("../01_Data/features")
features_path.mkdir(parents=True, exist_ok=True)

sales = pd.read_csv(data_path / "sales_clean.csv", parse_dates=["order_date"])
orders = pd.read_csv(
    data_path / "orders_clean.csv", parse_dates=["order_date"]
).drop(columns="Unnamed: 0", errors="ignore")
customers = pd.read_csv(
    data_path / "customers_clean.csv", parse_dates=["birth_date", "registration_date"]
).drop(columns="Unnamed: 0", errors="ignore")
stores = pd.read_csv(data_path / "stores_clean.csv").drop(columns="Unnamed: 0", errors="ignore")

print("Sales lines:", sales.shape)
print("All orders (every status):", orders.shape)
print("Customers:", customers.shape)
print("Stores:", stores.shape)

Sales lines: (22075, 23)
All orders (every status): (12000, 10)
Customers: (3000, 10)
Stores: (15, 8)


### Reference date

Customer measures such as recency and tenure need a fixed "as of" date. We use the **last day in the data, 31 December 2024**, so every customer is measured at the same point in time.

In [2]:
reference_date = pd.Timestamp("2024-12-31")
print("Last sale in the data:", sales["order_date"].max().date())
print("Reference date:", reference_date.date())

Last sale in the data: 2024-12-31
Reference date: 2024-12-31


# Customer Features

The segmentation needs to describe each customer along a few distinct dimensions:

| Dimension | Question it answers | Features |
|---|---|---|
| **Value** | How much do they bring in, and how profitably? | `total_revenue`, `total_profit`, `profit_margin`, `average_order_value` |
| **Engagement** | How often and how recently do they buy? | `orders`, `recency_days`, `tenure_days`, `orders_per_year` |
| **Product mix** | What do they buy? | `electronics_share`, `subcategories_bought` |
| **Price sensitivity** | Do they rely on discounts? | `average_discount`, `deep_discount_share` |
| **Service experience** | Do their orders go wrong? | `return_rate`, `cancellation_rate` |

We also carry descriptive attributes (gender, age, city, region, loyalty tier). These are used to **describe** the segments afterwards, not to create them, because we want segments based on behaviour rather than demographics.

## Value, engagement and product mix

These come from qualifying sales only.

In [3]:
import numpy as np

line_level = sales.assign(
    electronics_revenue=np.where(sales["category"] == "Electronics", sales["line_revenue"], 0.0),
    deep_discount_revenue=np.where(sales["discount_pct"] >= 20, sales["line_revenue"], 0.0),
    discount_x_revenue=sales["discount_pct"] * sales["line_revenue"],
)

customer_sales = line_level.groupby("customer_id").agg(
    first_purchase=("order_date", "min"),
    last_purchase=("order_date", "max"),
    orders=("order_id", "nunique"),
    total_revenue=("line_revenue", "sum"),
    total_profit=("line_profit", "sum"),
    electronics_revenue=("electronics_revenue", "sum"),
    deep_discount_revenue=("deep_discount_revenue", "sum"),
    discount_x_revenue=("discount_x_revenue", "sum"),
    subcategories_bought=("subcategory", "nunique"),
)

customer_sales["profit_margin"] = customer_sales["total_profit"] / customer_sales["total_revenue"]
customer_sales["average_order_value"] = customer_sales["total_revenue"] / customer_sales["orders"]
customer_sales["recency_days"] = (reference_date - customer_sales["last_purchase"]).dt.days
customer_sales["tenure_days"] = (reference_date - customer_sales["first_purchase"]).dt.days
customer_sales["electronics_share"] = customer_sales["electronics_revenue"] / customer_sales["total_revenue"]
customer_sales["deep_discount_share"] = customer_sales["deep_discount_revenue"] / customer_sales["total_revenue"]

# Revenue-weighted average discount: a 30% discount on a laptop matters more than on a T-shirt
customer_sales["average_discount"] = customer_sales["discount_x_revenue"] / customer_sales["total_revenue"]

print("Customers with at least one sale:", len(customer_sales))
customer_sales.head()

Customers with at least one sale: 2908


,first_purchase,last_purchase,orders,total_revenue,total_profit,electronics_revenue,deep_discount_revenue,discount_x_revenue,subcategories_bought,profit_margin,average_order_value,recency_days,tenure_days,electronics_share,deep_discount_share,average_discount
customer_id,,,,,,,,,,,,,,,,
CUS00001,2022-10-05,2024-11-10,4,26189.0,2899.0,23290.0,11200.0,346865.0,8,0.110695,6547.250000,51,818,0.889305,0.42766,13.244683
CUS00002,2023-01-02,2024-10-06,5,3522.0,1432.0,800.0,745.5,27045.0,5,0.406587,704.400000,86,729,0.227144,0.21167,7.678876
CUS00003,2023-04-06,2023-04-06,1,199.5,99.5,0.0,0.0,997.5,1,0.498747,199.500000,635,635,0.000000,0.00000,5.000000
CUS00004,2022-08-23,2024-06-02,6,27413.5,4163.5,22590.0,7721.0,282517.5,7,0.151878,4568.916667,212,861,0.824047,0.28165,10.305780
CUS00005,2024-06-19,2024-06-19,1,1340.0,670.0,0.0,0.0,0.0,2,0.500000,1340.000000,195,195,0.000000,0.00000,0.000000


A few design choices:

- **Tenure is measured from the first purchase**, not the registration date. The data quality log showed that 17% of order value comes from orders placed before registration, so registration is not a reliable "first contact" date.
- **Average discount is weighted by revenue.** A simple average would treat a 30% discount on a EGP 90 T-shirt the same as on a EGP 15,000 phone.
- **Deep-discount share** is the share of a customer's spend made at 20% off or more, the level at which Electronics stop making money.

### Purchase frequency

`orders` alone favours long-standing customers. `orders_per_year` adjusts for how long the customer has been buying. We use a minimum window of 90 days, so a customer whose first purchase was last week doesn't get an inflated yearly rate.

In [4]:
active_years = np.maximum(customer_sales["tenure_days"], 90) / 365.25
customer_sales["orders_per_year"] = customer_sales["orders"] / active_years

customer_sales["orders_per_year"].describe()

count    2908.000000
mean        1.740641
std         0.932483
min         0.334172
25%         1.120399
50%         1.583168
75%         2.121491
max        12.175000
Name: orders_per_year, dtype: float64

## Service experience

Returns and cancellations are not sales, so we take them from the full orders table, which includes every status. We look up each returned order's outcome so that **rejected** returns, which the business rules count as sales, are not counted as returns here.

In [5]:
returns = pd.read_csv(data_path / "returns_clean.csv").drop(columns="Unnamed: 0", errors="ignore")

orders_with_outcome = orders.merge(
    returns[["order_id", "return_status"]], on="order_id", how="left", validate="one_to_one"
)
orders_with_outcome["was_returned"] = (
    (orders_with_outcome["order_status"] == "Returned")
    & (orders_with_outcome["return_status"] != "Rejected")
)
orders_with_outcome["was_cancelled"] = orders_with_outcome["order_status"] == "Cancelled"

service = orders_with_outcome.groupby("customer_id").agg(
    all_orders=("order_id", "nunique"),
    returned_orders=("was_returned", "sum"),
    cancelled_orders=("was_cancelled", "sum"),
)
service["return_rate"] = service["returned_orders"] / service["all_orders"]
service["cancellation_rate"] = service["cancelled_orders"] / service["all_orders"]

service[["return_rate", "cancellation_rate"]].describe()

,return_rate,cancellation_rate
count,2945.000000,2945.000000
mean,0.079258,0.041621
std,0.154388,0.116276
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.142857,0.000000
max,1.000000,1.000000


## Descriptive attributes

Age is calculated at the reference date. Birth dates that failed validation during cleaning stay missing rather than being guessed.

In [6]:
attributes = customers[["customer_id", "gender", "city", "region", "loyalty_tier", "birth_date"]].copy()
attributes["age"] = np.floor((reference_date - attributes["birth_date"]).dt.days / 365.25)
attributes = attributes.drop(columns="birth_date")

print("Customers with unknown age:", attributes["age"].isna().sum())
attributes["age"].describe()

Customers with unknown age: 560


count    2440.000000
mean       37.432787
std         9.718696
min        21.000000
25%        29.000000
50%        37.000000
75%        46.000000
max        54.000000
Name: age, dtype: float64

## Assembling the customer table

We start from **all 3,000 customers**, so those with no qualifying sale are kept and flagged rather than silently dropped.

In [7]:
customer_columns = [
    "orders", "orders_per_year", "recency_days", "tenure_days",
    "total_revenue", "total_profit", "profit_margin", "average_order_value",
    "electronics_share", "subcategories_bought",
    "average_discount", "deep_discount_share",
]

customer_features = (
    attributes
    .merge(customer_sales[customer_columns], left_on="customer_id", right_index=True,
           how="left", validate="one_to_one")
    .merge(service[["all_orders", "return_rate", "cancellation_rate"]], left_on="customer_id",
           right_index=True, how="left", validate="one_to_one")
)

customer_features["has_sale"] = customer_features["orders"].notna()

# Customers with no sale: counts and amounts are genuinely zero
zero_columns = ["orders", "orders_per_year", "total_revenue", "total_profit",
                "subcategories_bought", "all_orders"]
customer_features[zero_columns] = customer_features[zero_columns].fillna(0)

# Customers who never ordered at all have no return or cancellation history
customer_features[["return_rate", "cancellation_rate"]] = (
    customer_features[["return_rate", "cancellation_rate"]].fillna(0)
)

customer_features = customer_features[
    ["customer_id", "has_sale"] + customer_columns
    + ["all_orders", "return_rate", "cancellation_rate",
       "gender", "age", "city", "region", "loyalty_tier"]
]

print("Customer table:", customer_features.shape)
print("Customers with a sale:", customer_features["has_sale"].sum())
print("Customers without a sale:", (~customer_features["has_sale"]).sum())
customer_features.head()

Customer table: (3000, 22)
Customers with a sale: 2908
Customers without a sale: 92


,customer_id,has_sale,orders,orders_per_year,recency_days,tenure_days,total_revenue,total_profit,profit_margin,average_order_value,...,average_discount,deep_discount_share,all_orders,return_rate,cancellation_rate,gender,age,city,region,loyalty_tier
0,CUS01518,True,2.0,0.866548,787.0,843.0,742.0,362.0,0.487871,371.000000,...,7.061995,0.000000,3.0,0.333333,0.000000,Male,43.0,Suez,Canal Zone,Bronze
1,CUS02370,True,6.0,2.036710,38.0,1076.0,12000.5,5730.5,0.477522,2000.083333,...,5.970376,0.009375,7.0,0.000000,0.142857,Male,21.0,Zagazig,Nile Delta,Bronze
2,CUS01962,True,2.0,1.272648,362.0,574.0,1394.5,664.5,0.476515,697.250000,...,8.689494,0.000000,2.0,0.000000,0.000000,Male,53.0,Giza,Greater Cairo,Gold
3,CUS00344,True,5.0,2.805300,115.0,651.0,1344.0,634.0,0.471726,268.800000,...,1.785714,0.000000,7.0,0.142857,0.142857,Female,24.0,Giza,Greater Cairo,Bronze
4,CUS02664,True,4.0,1.469819,229.0,994.0,3651.5,1771.5,0.485143,912.875000,...,6.152951,0.000000,5.0,0.200000,0.000000,Male,26.0,Tanta,Nile Delta,Bronze


For customers without a sale, ratio and time measures such as `profit_margin`, `recency_days` and `electronics_share` stay **missing**, because they are undefined rather than zero. These customers are excluded from the clustering and reported separately as **"Registered, never purchased"**.

### Validating the customer table

In [8]:
with_sale = customer_features[customer_features["has_sale"]]

print("Duplicate customer IDs:", customer_features["customer_id"].duplicated().sum())
print(f"Revenue matches sales table: "
      f"{np.isclose(with_sale['total_revenue'].sum(), sales['line_revenue'].sum())}")
print(f"Profit matches sales table:  "
      f"{np.isclose(with_sale['total_profit'].sum(), sales['line_profit'].sum())}")
print("Shares outside 0–1:",
      ((with_sale[["electronics_share", "deep_discount_share", "return_rate",
                   "cancellation_rate"]] < 0)
       | (with_sale[["electronics_share", "deep_discount_share", "return_rate",
                     "cancellation_rate"]] > 1)).sum().sum())
print("Negative recency or tenure:",
      ((with_sale["recency_days"] < 0) | (with_sale["tenure_days"] < 0)).sum())

print("\nMissing values among customers with a sale:")
print(with_sale.isna().sum().loc[lambda s: s > 0])

Duplicate customer IDs: 0
Revenue matches sales table: True
Profit matches sales table:  True
Shares outside 0–1: 0
Negative recency or tenure: 0

Missing values among customers with a sale:
age    544
dtype: int64


Every check passes. Customers with a sale have no missing values, apart from age for customers whose birth date was invalid. Age is a descriptive attribute only, so this doesn't affect the clustering.

In [9]:
with_sale[customer_columns + ["return_rate", "cancellation_rate"]].describe().T

,count,mean,std,min,25%,50%,75%,max
orders,2908.0,3.620702,1.748618,1.000000,2.000000,3.000000,5.000000,11.000000
orders_per_year,2908.0,1.740641,0.932483,0.334172,1.120399,1.583168,2.121491,12.175000
recency_days,2908.0,272.701169,243.407811,0.000000,83.000000,200.000000,397.000000,1093.000000
tenure_days,2908.0,816.217331,245.007437,4.000000,684.000000,885.000000,1007.000000,1095.000000
total_revenue,2908.0,13774.518741,17657.202822,67.500000,2473.625000,6254.250000,18497.875000,123695.000000
total_profit,2908.0,2929.484354,2991.325667,-3682.500000,1003.250000,2055.500000,3777.375000,26798.000000
profit_margin,2908.0,0.338424,0.149122,-0.145686,0.209878,0.381840,0.477321,0.571429
average_order_value,2908.0,3758.258347,4806.059486,67.500000,862.958333,1710.200000,4812.136905,55747.000000
electronics_share,2908.0,0.394711,0.404463,0.000000,0.000000,0.260483,0.834764,1.000000
subcategories_bought,2908.0,4.338033,1.734602,1.000000,3.000000,4.000000,6.000000,10.000000


**What the summary shows**

- **Profit margin** ranges from −15% to 57%. **27 customers are loss-making overall**: on average, about 90% of their spend was on Electronics bought at 20% off or more. These are exactly the customers a discount policy should be watching.
- **Electronics share**, **deep-discount share** and **profit margin** are all on a 0–1 scale, while revenue runs into the hundreds of thousands. The segmentation will therefore need to **scale** the features, and **log-transform** revenue, so no single feature dominates the clustering.

# Order Features

The loss-making order model must use **only information available at checkout**: what is in the basket, the discounts applied, and where and how the customer is paying.

We deliberately **leave out product cost and profit as features**. Profit is revenue minus cost, so giving the model cost would let it calculate the answer instead of learning which baskets and discounts are risky. Cost and profit are kept only to define the target.

| Group | Features |
|---|---|
| Basket size | `lines`, `items` |
| Basket value | `list_value` (before discount), `order_value` (after discount) |
| Basket mix | `electronics_items`, `clothing_items`, `electronics_value_share` |
| Discounts | `max_discount`, `max_electronics_discount`, `average_discount`, `electronics_value_at_20pct_plus` |
| Context | `store_type`, `payment_method`, `loyalty_tier`, `order_month` |
| Target (not a feature) | `loss_making`, with `order_profit` kept for reference |

In [10]:
line_values = sales.assign(
    list_value=sales["unit_price"] * sales["quantity"],
    is_electronics=sales["category"] == "Electronics",
)
line_values = line_values.assign(
    electronics_items=np.where(line_values["is_electronics"], line_values["quantity"], 0),
    clothing_items=np.where(~line_values["is_electronics"], line_values["quantity"], 0),
    electronics_list_value=np.where(line_values["is_electronics"], line_values["list_value"], 0.0),
    electronics_discount=line_values["discount_pct"].where(line_values["is_electronics"]),
    electronics_value_at_20pct_plus=np.where(
        line_values["is_electronics"] & (line_values["discount_pct"] >= 20),
        line_values["list_value"], 0.0),
    discount_x_list_value=line_values["discount_pct"] * line_values["list_value"],
)

order_features = line_values.groupby("order_id").agg(
    order_date=("order_date", "first"),
    customer_id=("customer_id", "first"),
    store_id=("store_id", "first"),
    payment_method=("payment_method", "first"),
    lines=("detail_id", "size"),
    items=("quantity", "sum"),
    electronics_items=("electronics_items", "sum"),
    clothing_items=("clothing_items", "sum"),
    list_value=("list_value", "sum"),
    order_value=("line_revenue", "sum"),
    electronics_list_value=("electronics_list_value", "sum"),
    electronics_value_at_20pct_plus=("electronics_value_at_20pct_plus", "sum"),
    discount_x_list_value=("discount_x_list_value", "sum"),
    max_discount=("discount_pct", "max"),
    max_electronics_discount=("electronics_discount", "max"),
    order_cost=("line_cost", "sum"),
    order_profit=("line_profit", "sum"),
).reset_index()

order_features["electronics_value_share"] = (
    order_features["electronics_list_value"] / order_features["list_value"]
)
order_features["average_discount"] = (
    order_features["discount_x_list_value"] / order_features["list_value"]
)
# Orders with no Electronics item: no Electronics discount was given
order_features["max_electronics_discount"] = order_features["max_electronics_discount"].fillna(0)

print("Order table:", order_features.shape)
order_features.head()

Order table: (10529, 20)


,order_id,order_date,customer_id,store_id,payment_method,lines,items,electronics_items,clothing_items,list_value,order_value,electronics_list_value,electronics_value_at_20pct_plus,discount_x_list_value,max_discount,max_electronics_discount,order_cost,order_profit,electronics_value_share,average_discount
0,ORD0000001,2023-02-28,CUS02412,7,Vodafone Cash,4,4,0,4,1360,1261.5,0.0,0.0,9850,25,0.0,650,611.5,0.000000,7.242647
1,ORD0000002,2023-12-10,CUS01109,11,Cash,2,4,0,4,2060,2020.0,0.0,0.0,4000,10,0.0,1020,1000.0,0.000000,1.941748
2,ORD0000003,2023-10-08,CUS01275,2,Fawry,1,1,0,1,430,322.5,0.0,0.0,10750,25,0.0,210,112.5,0.000000,25.000000
3,ORD0000004,2022-10-25,CUS01046,7,InstaPay,5,6,0,6,1280,1190.0,0.0,0.0,9000,15,0.0,600,590.0,0.000000,7.031250
4,ORD0000005,2024-03-07,CUS00418,6,Fawry,3,4,3,1,9810,9530.0,9600.0,0.0,28000,5,5.0,7800,1730.0,0.978593,2.854230


`average_discount` is weighted by **list value**, so it reflects how much of the basket's value was discounted.

### Context known at checkout

We add the store type, the customer's loyalty tier and the month of the order. The loyalty tier is the customer's tier as recorded in the customer table.

In [11]:
order_features = (
    order_features
    .merge(stores[["store_id", "store_type"]], on="store_id", how="left", validate="many_to_one")
    .merge(customers[["customer_id", "loyalty_tier"]], on="customer_id", how="left",
           validate="many_to_one")
)
order_features["order_month"] = order_features["order_date"].dt.month
order_features["order_year"] = order_features["order_date"].dt.year

### Target and chronological split

- **Target:** `loss_making` = 1 when the order's cost exceeds its revenue.
- **Split:** orders from 2022–2023 are for training and orders from 2024 are for testing, as set in the project charter. We record the split in the table so every modelling notebook uses exactly the same one.

In [12]:
order_features["loss_making"] = (order_features["order_profit"] < 0).astype(int)
order_features["dataset"] = np.where(order_features["order_year"] <= 2023, "train", "test")

order_features = order_features[[
    "order_id", "order_date", "order_year", "dataset", "customer_id", "store_id",
    # features known at checkout
    "lines", "items", "electronics_items", "clothing_items",
    "list_value", "order_value", "electronics_value_share",
    "max_discount", "max_electronics_discount", "average_discount",
    "electronics_value_at_20pct_plus",
    "store_type", "payment_method", "loyalty_tier", "order_month",
    # reference and target (never used as features)
    "order_cost", "order_profit", "loss_making",
]]

(
    order_features.groupby("dataset")
    .agg(orders=("order_id", "size"),
         loss_making_orders=("loss_making", "sum"),
         loss_rate=("loss_making", "mean"))
    .loc[["train", "test"]]
)

,orders,loss_making_orders,loss_rate
dataset,,,
train,6993,86,0.012298
test,3536,40,0.011312


The target is rare: about **1 in 85 orders** loses money, in both periods. The test set has **40 loss-making orders**. That is enough to judge recall, but each missed order moves recall by about 2.5 percentage points, which we'll bear in mind when reading the results.

### Validating the order table

In [13]:
print("Duplicate order IDs:", order_features["order_id"].duplicated().sum())
print("Orders match sales table:", len(order_features) == sales["order_id"].nunique())
print(f"Order value matches sales revenue: "
      f"{np.isclose(order_features['order_value'].sum(), sales['line_revenue'].sum())}")
print(f"Order profit matches sales profit: "
      f"{np.isclose(order_features['order_profit'].sum(), sales['line_profit'].sum())}")
print("Items = Electronics + Clothing items:",
      (order_features["items"] == order_features["electronics_items"]
       + order_features["clothing_items"]).all())
print("Order value never above list value:",
      (order_features["order_value"] <= order_features["list_value"] + 0.01).all())
print("\nMissing values:", order_features.isna().sum().sum())

Duplicate order IDs: 0
Orders match sales table: True
Order value matches sales revenue: True
Order profit matches sales profit: True
Items = Electronics + Clothing items: True
Order value never above list value: True

Missing values: 0


All checks pass and there are no missing values.

### A first look at the most important feature

The exploratory analysis suggested that the highest Electronics discount in a basket is what drives losses. We confirm that the feature captures this before modelling.

In [14]:
(
    order_features
    .groupby("max_electronics_discount")
    .agg(orders=("order_id", "size"),
         loss_making_orders=("loss_making", "sum"),
         loss_rate=("loss_making", "mean"))
    .style.format({"loss_rate": "{:.1%}"})
)

,orders,loss_making_orders,loss_rate
max_electronics_discount,,,
0.000000,9094,0,0.0%
5.000000,339,0,0.0%
10.000000,513,0,0.0%
15.000000,240,0,0.0%
20.000000,178,16,9.0%
25.000000,126,78,61.9%
30.000000,39,32,82.1%


The pattern from the exploratory analysis carries through: **no order loses money unless an Electronics item is discounted 20% or more**, and the loss rate climbs steeply from there. The model will need to learn *when* a deep Electronics discount is rescued by the rest of the basket. That is why the basket value and mix features matter.

# Saving the Feature Tables

In [15]:
customer_features.to_csv(features_path / "customer_features.csv", index=False)
order_features.to_csv(features_path / "order_features.csv", index=False)

print("Saved customer_features.csv:", customer_features.shape)
print("Saved order_features.csv:", order_features.shape)

Saved customer_features.csv: (3000, 22)
Saved order_features.csv: (10529, 24)


# Feature Engineering Complete

| Output | Rows | Key contents | Next notebook |
|---|---:|---|---|
| `01_Data/features/customer_features.csv` | 3,000 customers | Value, engagement, product mix, price sensitivity, service experience, plus descriptive attributes | `05_customer_segmentation.ipynb` |
| `01_Data/features/order_features.csv` | 10,529 orders | Basket size, value, mix and discounts known at checkout; the `loss_making` target; train/test split | `06_loss_making_orders.ipynb` |

**Decisions carried forward:**

- Customers without a sale are kept but flagged with `has_sale = False`, and excluded from clustering.
- Revenue will be log-transformed and all clustering features scaled.
- Cost and profit are never used as order features, only to define the target.
- Every model uses the same train (2022–2023) / test (2024) split recorded in `dataset`.